# Exercise-variant generation evaluation

Driver for the harness. Every cell is thin on purpose — the logic lives in the modules next to this
notebook, so a crashed kernel loses nothing and each stage can be rerun independently over stored data.

Order: corpus → rounds → timings → checks → tables → figures → rubric sampling.

Prerequisites: `config.ini` filled in, the instance running (`local/eval/start-server.sh`), and
`corpus/create_corpus.py` run once.

In [ ]:
import importlib, json, os

import analysis, artifacts, checks, logs, matrix, runner, utils
from run_evaluation import prompt_commit_sha

for module in (logs, checks, analysis, artifacts, runner):
    importlib.reload(module)

RESULTS = "results"          # use "results-pilot" for pre-freeze pilot runs — never mix the two
SOURCES = os.path.join("corpus", "sources")

session = utils.authenticated_session()
corpus = runner.load_corpus()
print(json.dumps(corpus, indent=2, sort_keys=True))
print("prompt commit:", prompt_commit_sha())

## Rounds

One round is one replicate of every configuration on both exercise types. Stopping after any completed
round leaves a balanced corpus with equal n everywhere; extending is just running more rounds. Resume is
automatic — completed `run_id`s are skipped.

Job records have a 24-hour TTL, so rounds must run in contiguous blocks: a round paused overnight cannot
be backfilled and its runs have to be repeated.

In [ ]:
harness = runner.Harness(session, RESULTS, prompt_commit_sha(), concurrency=utils.CONCURRENCY)

TARGET_ROUNDS = 4  # floor; raise and rerun this cell to extend
for round_number in range(1, TARGET_ROUNDS + 1):
    harness.run_round(round_number, [c.config_id for c in matrix.CONFIGURATIONS], matrix.EXERCISE_TYPES)

In [ ]:
# Serial timing subset: reported separately, because contention on the single build agent inflates wall
# time and would make the phase breakdown dishonest.
serial_harness = runner.Harness(session, RESULTS, prompt_commit_sha(), concurrency=1)
for exercise_type in matrix.EXERCISE_TYPES:
    for round_number in (1, 2):
        serial_harness.run_one(exercise_type, "C3", round_number, serial=True)

## Timings from the log

Separate cell so it reruns over stored logs without touching the instance. Fails loudly on an incomplete
timeline rather than dropping it quietly, and sanity-checks that summed phase durations reconcile with
`finished_at - started_at`.

In [ ]:
runs = analysis.load_runs(RESULTS)
print(f"{len(runs)} runs in the ledger")

incomplete = [r["run_id"] for r in runs if not r.get("phase_timeline_complete")]
if incomplete:
    print("INCOMPLETE TIMELINES (investigate, do not silently drop):")
    for run in runs:
        if not run.get("phase_timeline_complete"):
            print(" ", run["run_id"], "->", run.get("phase_timeline_problem"))

for run in runs:
    durations = run.get("phase_durations_seconds") or {}
    wall = run.get("wall_seconds")
    if wall and durations and abs(sum(durations.values()) - wall) > 2:
        print(f"RECONCILIATION GAP {run['run_id']}: phases={sum(durations.values()):.1f}s wall={wall:.1f}s")

# QUEUED should be near zero: the executor is core 4 / max 8 / queue 32, so at these concurrencies nothing
# should queue. A large value means the harness outran the pool and the wall-time numbers are contaminated.
queued = [(r["run_id"], (r.get("phase_durations_seconds") or {}).get("QUEUED", 0.0)) for r in runs]
worst = sorted(queued, key=lambda pair: -pair[1])[:5]
print("largest QUEUED segments:", worst)

## Automated checks

Run over every surviving variant at no marginal cost. These carry the mechanical criteria so the rubric's
reading attention goes to what no check can reach.

In [ ]:
check_results = [checks.run_checks_for_run(run, os.path.join(RESULTS, "artifacts"), SOURCES) for run in runs]
with open(os.path.join(RESULTS, "checks.jsonl"), "w", encoding="utf-8") as handle:
    for result in check_results:
        handle.write(json.dumps(result, sort_keys=True) + "\n")

programming = [c for c in check_results if c.get("has_artifacts") and c.get("exercise_type") == "programming"]
print("programming variants checked:", len(programming))
print("  with stray <testid>:", sum(1 for c in programming if c["statement"]["has_stray_testid"]))
print("  with dangling task refs:", sum(1 for c in programming if c["statement"]["has_dangling_task_reference"]))
print("  with fenced PlantUML:", sum(1 for c in programming if c["statement"]["has_fenced_plantuml"]))

## Tables and figures

Rates carry their n and a Wilson 95 % interval, never a bare percentage. Pooling is used for headline
reliability only — the narrative sweeps are read cell by cell, because pooling is exactly what would erase
the differences those cells exist to show.

In [ ]:
rubric_path = os.path.join(RESULTS, "rubric.jsonl")
rubric_scores = []
if os.path.exists(rubric_path):
    with open(rubric_path, encoding="utf-8") as handle:
        rubric_scores = [json.loads(line) for line in handle if line.strip()]

outcomes = analysis.outcomes_table(runs, RESULTS)
cost = analysis.cost_table(runs, RESULTS)
quality = analysis.quality_table(runs, check_results, rubric_scores, RESULTS)
written = analysis.figures(runs, RESULTS)
print(f"tables: {len(outcomes)} outcome rows, {len(cost)} cost rows, {len(quality)} quality rows")
print("figures:", written)

## Rubric sampling

Draws the variants to score, per the frozen rule in `rubric.md`: layer 1 is one survivor per configuration
per exercise type, scored as a complete balanced layer first; layer 2 adds a second for C9–C12 (the
unconfounded narrative sweep) and C4/C13 (where the domain supplies no ordering key), which is where
reading is the only available instrument.

The scoring itself is done by the agent against `rubric.md`, one variant at a time, appending to
`rubric.jsonl` immediately so an interrupted session resumes by checking which run ids are already scored.

In [ ]:
import random
from collections import defaultdict

LAYER_2_CONFIGS = ("C9", "C10", "C11", "C12", "C4", "C13")
survivors = defaultdict(list)
for run in runs:
    if run["terminal_phase"] in analysis.SURVIVING_PHASES and not run.get("serial"):
        survivors[(run["exercise_type"], run["config_id"])].append(run["run_id"])

rng = random.Random(20260805)
sample = {"layer_1": [], "layer_2": [], "short_cells": []}
for key in sorted(survivors):
    available = sorted(survivors[key])
    rng.shuffle(available)
    wanted = 2 if key[1] in LAYER_2_CONFIGS else 1
    if len(available) < wanted:
        sample["short_cells"].append({"cell": key, "wanted": wanted, "available": len(available)})
    sample["layer_1"].extend(available[:1])
    sample["layer_2"].extend(available[1:wanted])

already_scored = {score["run_id"] for score in rubric_scores if score.get("scorer") == "primary"}
print("layer 1:", len(sample["layer_1"]), "| layer 2:", len(sample["layer_2"]), "| already scored:", len(already_scored))
print("cells with fewer survivors than wanted (a result in itself):", sample["short_cells"])
with open(os.path.join(RESULTS, "rubric-sample.json"), "w", encoding="utf-8") as handle:
    json.dump(sample, handle, indent=2, sort_keys=True)